In [ ]:
#Load and insepct the booking records.

import numpy as np
import pandas as pd

df = pd.read_csv("13_hotel_bookings.csv")

print("Dataset shape (rows, columns):", df.shape)
print("\nColumn names aur data types:\n", df.dtypes)
print("\nPehle 5 records:\n", df.head())
print("\nSummary statistics:\n", df.describe(include="all"))

Dataset shape (rows, columns): (761, 9)

Column names aur data types:
 Booking_ID          object
Checkin_Date        object
Lead_Time_Days     float64
Room_Type           object
Room_Rate          float64
Nights               int64
Booking_Channel     object
Traveler_Type       object
Canceled            object
dtype: object

Pehle 5 records:
   Booking_ID Checkin_Date  Lead_Time_Days Room_Type  Room_Rate  Nights  \
0     B00000   2025-05-15             NaN    Deluxe    3080.04       6   
1     B00001   2025-10-14           123.0    Deluxe    6418.87       2   
2     B00002   2026-08-08           103.0  Standard   14073.29       2   
3     B00003   2026-03-20            93.0    Deluxe   14371.81       6   
4     B00004   2025-05-15           159.0  Standard   14898.76       7   

  Booking_Channel Traveler_Type Canceled  
0  Direct Website      Business       No  
1    Travel Agent       Leisure       No  
2    Travel Agent       Leisure       No  
3       Corporate       Leisure     

In [ ]:
# Identity missing values and duplicate Booking_ID records.

print("Missing values per column:\n", df.isnull().sum())

dup_count = df["Booking_ID"].duplicated().sum()
print(f"\nDuplicate Booking_ID rows ki sankhya: {dup_count}")

dup_rows = df[df.duplicated(subset=["Booking_ID"], keep=False)].sort_values("Booking_ID")
print("\nDuplicate Booking_ID records:\n", dup_rows.to_string())

# Duplicate Booking_IDs hata kar clean dataframe banayein
df = df.drop_duplicates(subset=["Booking_ID"], keep="first").reset_index(drop=True)
print("\nDeduplication ke baad shape:", df.shape)

Missing values per column:
 Booking_ID            0
Checkin_Date          0
Lead_Time_Days        0
Room_Type             0
Room_Rate             0
Nights                0
Booking_Channel       0
Traveler_Type         0
Canceled              0
Arrival_Month_Num     0
Arrival_Month         0
Arrival_YearMonth     0
Arrival_Weekday       0
Booking_Revenue       0
Lead_Time_Band        0
Lead_Time_Category    0
dtype: int64

Duplicate Booking_ID rows ki sankhya: 0

Duplicate Booking_ID records:
 Empty DataFrame
Columns: [Booking_ID, Checkin_Date, Lead_Time_Days, Room_Type, Room_Rate, Nights, Booking_Channel, Traveler_Type, Canceled, Arrival_Month_Num, Arrival_Month, Arrival_YearMonth, Arrival_Weekday, Booking_Revenue, Lead_Time_Band, Lead_Time_Category]
Index: []

Deduplication ke baad shape: (750, 16)


In [ ]:
# Room_Rate ke missing values ko Room_Type ke median se fill karna (kyunki rate room type ke hisaab se badalta hai)
df["Room_Rate"] = df["Room_Rate"].fillna(
    df.groupby("Room_Type")["Room_Rate"].transform("median")
)

# Lead_Time_Days ke missing values ko Booking_Channel aur Traveler_Type ke median se fill karna
df["Lead_Time_Days"] = df["Lead_Time_Days"].fillna(
    df.groupby(["Booking_Channel", "Traveler_Type"])["Lead_Time_Days"].transform("median")
)

# Fallback safety median
df["Room_Rate"] = df["Room_Rate"].fillna(df["Room_Rate"].median())
df["Lead_Time_Days"] = df["Lead_Time_Days"].fillna(df["Lead_Time_Days"].median())

print("Imputation ke baad bache missing values:\n", df.isnull().sum())

Imputation ke baad bache missing values:
 Booking_ID         0
Checkin_Date       0
Lead_Time_Days     0
Room_Type          0
Room_Rate          0
Nights             0
Booking_Channel    0
Traveler_Type      0
Canceled           0
dtype: int64


In [ ]:
#Convert Checkin_Date to datetime and extract arrival month and weekday.

df["Checkin_Date"] = pd.to_datetime(df["Checkin_Date"])
df["Arrival_Month_Num"] = df["Checkin_Date"].dt.month
df["Arrival_Month"] = df["Checkin_Date"].dt.strftime("%b")
df["Arrival_YearMonth"] = df["Checkin_Date"].dt.to_period("M")
df["Arrival_Weekday"] = df["Checkin_Date"].dt.day_name()

print(df[["Booking_ID", "Checkin_Date", "Arrival_Month", "Arrival_YearMonth", "Arrival_Weekday"]].head())

  Booking_ID Checkin_Date Arrival_Month Arrival_YearMonth Arrival_Weekday
0     B00000   2025-05-15           May           2025-05        Thursday
1     B00001   2025-10-14           Oct           2025-10         Tuesday
2     B00002   2026-08-08           Aug           2026-08        Saturday
3     B00003   2026-03-20           Mar           2026-03          Friday
4     B00004   2025-05-15           May           2025-05        Thursday


In [ ]:
# Calculate total bookings, total room nights , and overall cancelation percentage.


total_bookings = len(df)
canceled_bookings = int((df["Canceled"] == "Yes").sum())
non_canceled_bookings = int((df["Canceled"] == "No").sum())
cancellation_pct = (canceled_bookings / total_bookings) * 100

total_room_nights = int(df["Nights"].sum())
realized_room_nights = int(df.loc[df["Canceled"] == "No", "Nights"].sum())

print(f"Total Bookings: {total_bookings}")
print(f"Non-Canceled Bookings: {non_canceled_bookings} | Canceled Bookings: {canceled_bookings}")
print(f"Overall Cancellation Percentage: {cancellation_pct:.2f}%")
print(f"Total Room Nights (All): {total_room_nights}")
print(f"Realized Room Nights (Non-Canceled): {realized_room_nights}")

Total Bookings: 750
Non-Canceled Bookings: 601 | Canceled Bookings: 149
Overall Cancellation Percentage: 19.87%
Total Room Nights (All): 2988
Realized Room Nights (Non-Canceled): 2393


In [ ]:
# Calculate row-wise total bookings, revenue (Night * Room_Rate) for non-canceled bookings.

df["Booking_Revenue"] = np.where(df["Canceled"] == "No", df["Nights"] * df["Room_Rate"], 0.0)

realized_df = df[df["Canceled"] == "No"].copy()
total_realized_revenue = realized_df["Booking_Revenue"].sum()

print(f"Total Realized Revenue: ₹{total_realized_revenue:,.2f}")
print("\nNon-canceled bookings ke revenue sample:\n", realized_df[["Booking_ID", "Room_Type", "Room_Rate", "Nights", "Booking_Revenue"]].head())

Total Realized Revenue: ₹21,080,871.94

Non-canceled bookings ke revenue sample:
   Booking_ID Room_Type  Room_Rate  Nights  Booking_Revenue
0     B00000    Deluxe    3080.04       6         18480.24
1     B00001    Deluxe    6418.87       2         12837.74
2     B00002  Standard   14073.29       2         28146.58
4     B00004  Standard   14898.76       7        104291.32
5     B00005  Standard    8455.03       2         16910.06


In [ ]:
#Compare average room  rates and occupancy across room types(e.g.,Standard,Deluxe,Suite).

room_summary = df.groupby("Room_Type").agg(
    Total_Bookings=("Booking_ID", "count"),
    Realized_Bookings=("Canceled", lambda s: (s == "No").sum()),
    Avg_Room_Rate=("Room_Rate", "mean"),
    Total_Room_Nights=("Nights", "sum"),
    Realized_Room_Nights=("Nights", lambda s: s[df.loc[s.index, "Canceled"] == "No"].sum()),
    Realized_Revenue=("Booking_Revenue", "sum")
)

room_summary["Realized_Nights_Share_Pct"] = (
    room_summary["Realized_Room_Nights"] / realized_room_nights * 100
)
room_summary["Realized_Revenue_Share_Pct"] = (
    room_summary["Realized_Revenue"] / total_realized_revenue * 100
)

print(room_summary.round(2).to_string())

           Total_Bookings  Realized_Bookings  Avg_Room_Rate  Total_Room_Nights  Realized_Room_Nights  Realized_Revenue  Realized_Nights_Share_Pct  Realized_Revenue_Share_Pct
Room_Type                                                                                                                                                                    
Deluxe                259                203        8430.98               1065                   835        7157340.20                      34.89                       33.95
Standard              383                311        8625.36               1482                  1203       10664972.53                      50.27                       50.59
Suite                 108                 87        9240.82                441                   355        3258559.21                      14.83                       15.46


In [ ]:
#Rank book channels (e.g., Direct Website ,OTA, Travel Agent) by booking volume and revenue.

channel_summary = df.groupby("Booking_Channel").agg(
    Total_Bookings=("Booking_ID", "count"),
    Realized_Bookings=("Canceled", lambda s: (s == "No").sum()),
    Realized_Room_Nights=("Nights", lambda s: s[df.loc[s.index, "Canceled"] == "No"].sum()),
    Avg_Room_Rate=("Room_Rate", "mean"),
    Realized_Revenue=("Booking_Revenue", "sum")
)

channel_summary["Volume_Rank"] = channel_summary["Total_Bookings"].rank(ascending=False, method="min").astype(int)
channel_summary["Revenue_Rank"] = channel_summary["Realized_Revenue"].rank(ascending=False, method="min").astype(int)
channel_summary = channel_summary.sort_values("Revenue_Rank")

print(channel_summary.round(2).to_string())

# Multi-level summary (Room_Type x Booking_Channel)
room_channel_summary = df.groupby(["Room_Type", "Booking_Channel"]).agg(
    Bookings=("Booking_ID", "count"),
    Realized_Nights=("Nights", lambda s: s[df.loc[s.index, "Canceled"] == "No"].sum()),
    Avg_Rate=("Room_Rate", "mean"),
    Realized_Revenue=("Booking_Revenue", "sum")
)
print("\nRoom_Type x Booking_Channel summary:\n", room_channel_summary.round(2).to_string())

                 Total_Bookings  Realized_Bookings  Realized_Room_Nights  Avg_Room_Rate  Realized_Revenue  Volume_Rank  Revenue_Rank
Booking_Channel                                                                                                                     
OTA                         181                154                   663        8491.07        5700983.18            3             1
Travel Agent                194                156                   587        8922.36        5356168.13            2             2
Corporate                   197                154                   591        8491.42        5212337.71            1             3
Direct Website              178                137                   552        8677.05        4811382.92            4             4

Room_Type x Booking_Channel summary:
                            Bookings  Realized_Nights  Avg_Rate  Realized_Revenue
Room_Type Booking_Channel                                                       
D

In [ ]:
# Agar Q14 se pehle run kar rahe hain to Lead_Time_Band yahan bhi create kar sakte hain
if "Lead_Time_Band" not in df.columns:
    df["Lead_Time_Band"] = pd.cut(
        df["Lead_Time_Days"],
        bins=[-0.1, 14, 60, np.inf],
        labels=["Last-Minute (0–14d)", "Short-Term (15–60d)", "Advance (61+d)"]
    )

cancel_by_channel = df.groupby("Booking_Channel")["Canceled"].apply(lambda s: (s == "Yes").mean() * 100)
print("Cancellation Rate (%) by Booking Channel:\n", cancel_by_channel.round(2))

cancel_by_lt = df.groupby("Lead_Time_Band", observed=False)["Canceled"].apply(lambda s: (s == "Yes").mean() * 100)
print("\nCancellation Rate (%) by Lead-Time Band:\n", cancel_by_lt.round(2))

cancel_matrix = df.pivot_table(
    index="Booking_Channel",
    columns="Lead_Time_Band",
    values="Canceled",
    aggfunc=lambda s: (s == "Yes").mean() * 100,
    observed=False
)
print("\nCancellation Rate (%) Matrix (Channel x Lead-Time Band):\n", cancel_matrix.round(2).to_string())

Cancellation Rate (%) by Booking Channel:
 Booking_Channel
Corporate         21.83
Direct Website    23.03
OTA               14.92
Travel Agent      19.59
Name: Canceled, dtype: float64

Cancellation Rate (%) by Lead-Time Band:
 Lead_Time_Band
Last-Minute (0–14d)    21.05
Short-Term (15–60d)    16.94
Advance (61+d)         20.78
Name: Canceled, dtype: float64

Cancellation Rate (%) Matrix (Channel x Lead-Time Band):
 Lead_Time_Band   Last-Minute (0–14d)  Short-Term (15–60d)  Advance (61+d)
Booking_Channel                                                          
Corporate                      33.33                25.00           19.86
Direct Website                 26.67                18.87           24.55
OTA                             9.09                 4.65           18.90
Travel Agent                   15.79                18.60           20.45


In [ ]:
monthly_rev = (
    df.groupby(["Arrival_Month_Num", "Arrival_Month"])
    .agg(
        Total_Bookings=("Booking_ID", "count"),
        Realized_Room_Nights=("Nights", lambda s: s[df.loc[s.index, "Canceled"] == "No"].sum()),
        Realized_Revenue=("Booking_Revenue", "sum")
    )
    .reset_index()
    .sort_values("Arrival_Month_Num")
)

top_3_months = monthly_rev.sort_values("Realized_Revenue", ascending=False).head(3)

print("Sabhi months ka summary:\n", monthly_rev.round(2).to_string(index=False))
print("\nTop 3 Peak Revenue Months:\n", top_3_months.round(2).to_string(index=False))

Sabhi months ka summary:
  Arrival_Month_Num Arrival_Month  Total_Bookings  Realized_Room_Nights  Realized_Revenue
                 1           Jan              77                   254        2230977.47
                 2           Feb              52                   151        1315006.12
                 3           Mar              54                   160        1403538.15
                 4           Apr              81                   265        2253945.94
                 5           May              77                   255        2385588.14
                 6           Jun              80                   238        2141920.37
                 7           Jul              63                   207        1792041.45
                 8           Aug              70                   231        2043432.54
                 9           Sep              72                   218        1735673.94
                10           Oct              72                   238        215919

In [ ]:
stay_comparison = df.groupby("Traveler_Type").agg(
    Total_Bookings=("Booking_ID", "count"),
    Avg_Stay_All_Nights=("Nights", "mean"),
    Median_Stay_All_Nights=("Nights", "median"),
    Avg_Stay_Realized_Nights=("Nights", lambda s: s[df.loc[s.index, "Canceled"] == "No"].mean()),
    Realized_Revenue=("Booking_Revenue", "sum")
)

print(stay_comparison.round(2).to_string())

               Total_Bookings  Avg_Stay_All_Nights  Median_Stay_All_Nights  Avg_Stay_Realized_Nights  Realized_Revenue
Traveler_Type                                                                                                         
Business                  383                 4.05                     4.0                      4.02       10736252.32
Leisure                   367                 3.92                     4.0                      3.94       10344619.62


In [ ]:
rates_arr = df["Room_Rate"].to_numpy()

rate_mean = np.mean(rates_arr)
rate_median = np.median(rates_arr)
rate_std = np.std(rates_arr, ddof=1)

print(f"Daily Room Rate Mean: ₹{rate_mean:,.2f}")
print(f"Daily Room Rate Median: ₹{rate_median:,.2f}")
print(f"Daily Room Rate Std Dev: ₹{rate_std:,.2f}")

Daily Room Rate Mean: ₹8,646.86
Daily Room Rate Median: ₹8,862.02
Daily Room Rate Std Dev: ₹3,590.27


In [ ]:
realized_revs = df.loc[df["Canceled"] == "No", "Booking_Revenue"].to_numpy()

q1, q3 = np.percentile(realized_revs, [25, 75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

high_outliers = df[(df["Canceled"] == "No") & (df["Booking_Revenue"] > upper_fence)].sort_values(
    "Booking_Revenue", ascending=False
)

print(f"Q1: ₹{q1:,.2f} | Q3: ₹{q3:,.2f} | IQR: ₹{iqr:,.2f}")
print(f"Upper Fence (Q3 + 1.5*IQR): ₹{upper_fence:,.2f}")
print(f"High booking value outliers ki sankhya: {len(high_outliers)}")
print("\nOutlier records:\n", high_outliers[["Booking_ID", "Checkin_Date", "Room_Type", "Room_Rate", "Nights", "Booking_Channel", "Traveler_Type", "Booking_Revenue"]].to_string(index=False))

Q1: ₹14,695.97 | Q3: ₹50,434.40 | IQR: ₹35,738.43
Upper Fence (Q3 + 1.5*IQR): ₹104,042.05
High booking value outliers ki sankhya: 1

Outlier records:
 Booking_ID Checkin_Date Room_Type  Room_Rate  Nights Booking_Channel Traveler_Type  Booking_Revenue
    B00004   2025-05-15  Standard   14898.76       7    Travel Agent      Business        104291.32


In [ ]:
df["Lead_Time_Category"] = pd.cut(
    df["Lead_Time_Days"],
    bins=[-0.1, 14, 60, np.inf],
    labels=["Last-Minute", "Short-Term", "Advance"]
)

lt_summary = df.groupby("Lead_Time_Category", observed=False).agg(
    Bookings=("Booking_ID", "count"),
    Avg_Lead_Time=("Lead_Time_Days", "mean"),
    Realized_Revenue=("Booking_Revenue", "sum")
)

print(lt_summary.round(2).to_string())

                    Bookings  Avg_Lead_Time  Realized_Revenue
Lead_Time_Category                                           
Last-Minute               57           6.21        1633659.12
Short-Term               183          38.36        5108620.50
Advance                  510         119.56       14338592.32


recommendations = [
    "1. Corporate aur Direct Website bookings par cancellation control badhayein: Corporate last-minute (0-14 din) bookings mein cancellation rate 33.33% aur Direct Website par overall cancellation rate 23.03% hai; 7-14 din ke andar partial advance ya strict cut-off rules lagayein.",
    "2. Peak months (May, April, January) aur Thursdays ke liye dynamic pricing aur staffing plan karein: May (₹23,85,588.14), April (₹22,53,945.94), aur January (₹22,30,977.47) top 3 months hain, aur Thursday sabse zyada bookings (128) aur realized nights (462) laata hai.",
    "3. Direct Website ka conversion aur guest retention sudharein: Direct Website revenue (₹48,11,382.92) aur bookings (178) mein 4th rank par hai aur cancellation 23.03% hai; late check-out, room upgrade priority, aur easy date-modification dein.",
    "4. High-realization channels (OTA aur Travel Agent) ko priority dein: OTA ka cancellation rate sabse kam (14.92%) aur realized revenue sabse zyada (₹57,00,983.18) hai, jabki Travel Agent ka average rate sabse uncha (₹8,922.36) hai.",
    "5. Low-revenue aur high-cancellation months ke liye targeted packages launch karein: November (₹7,60,400.02) aur December (₹8,59,157.00) mein low revenue aur Feb (23.08%) / Mar (25.93%) mein high cancellations ke liye 3-4 raaton ke packages chalaayein."
]

for r in recommendations:
    print(r)